In [1]:

import os
from pathlib import Path

# Update this to wherever your 4 folders live
data_dir = Path("../data")  

folders = [f for f in data_dir.iterdir() if f.is_dir()]
print(f"Found {len(folders)} folders:\n")
for folder in folders:
    pdf_files = list(folder.glob("*.pdf"))
    print(f"📁 {folder.name}: {len(pdf_files)} PDF(s)")
    for pdf in pdf_files:
        size_kb = pdf.stat().st_size / 1024
        print(f"   - {pdf.name} ({size_kb:.1f} KB)") # and here teh pdf name and size 
        #after conversion is printing
    print()

Found 4 folders:

📁 amazon: 7 PDF(s)
   - amazon 10-k 2023.pdf (781.8 KB)
   - amazon 10-k 2024.pdf (736.9 KB)
   - amazon 10-q q1 2024.pdf (478.2 KB)
   - amazon 10-q q1 2025.pdf (474.9 KB)
   - amazon 10-q q2 2024.pdf (497.4 KB)
   - amazon 10-q q2 2025.pdf (497.0 KB)
   - amazon 10-q q3 2024.pdf (1139.7 KB)

📁 apple: 6 PDF(s)
   - apple 10-k 2023.pdf (788.4 KB)
   - apple 10-k 2024.pdf (1068.2 KB)
   - apple 10-q q1 2024.pdf (274.2 KB)
   - apple 10-q q2 2024.pdf (325.6 KB)
   - apple 10-q q4 2023.pdf (265.4 KB)
   - apple 8-k q4 2023.pdf (99.0 KB)

📁 google: 7 PDF(s)
   - google 10-k 2023.pdf (1339.6 KB)
   - google 10-k 2024 (1).pdf (1308.9 KB)
   - google 10-k 2024.pdf (1308.9 KB)
   - google 10-q q1 2025.pdf (452.8 KB)
   - google 10-q q2 2024.pdf (498.3 KB)
   - google 10-q q2 2025.pdf (512.9 KB)
   - google 10-q q3 2024.pdf (508.6 KB)

📁 meta: 7 PDF(s)
   - meta 10-q q1 2024.pdf (240.1 KB)
   - meta 10-q q1 2025.pdf (228.6 KB)
   - meta 10-q q2 2024.pdf (148.5 KB)
   - meta 10

In [3]:
import re
def normalize_name(filename: str) -> str:
    """Strip things like ' (1)' before comparing, to catch duplicate downloads."""
    name = filename.lower()
    name = re.sub(r"\s*\(\d+\)", "", name)  # removes " (1)", " (2)" files
    return name.strip()


all_pdf_files = []
seen_normalized_names = set()
skipped_duplicates = []

folders = [f for f in data_dir.iterdir() if f.is_dir()]

for folder in folders:
    for pdf_file in folder.glob("*.pdf"):
        norm_name = normalize_name(pdf_file.name)
        if norm_name in seen_normalized_names:
            skipped_duplicates.append(pdf_file.name)
            continue
        seen_normalized_names.add(norm_name)
        all_pdf_files.append(pdf_file)

print(f"Total unique PDFs to process: {len(all_pdf_files)}")
print(f"Skipped duplicates: {skipped_duplicates}")

Total unique PDFs to process: 26
Skipped duplicates: ['google 10-k 2024.pdf']


In [4]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import PyMuPDFLoader
# def process_all_pdfs(pdf_files, chunk_size=1000, chunk_overlap=200):
#     text_splitter = RecursiveCharacterTextSplitter(
#         chunk_size=chunk_size,
#         chunk_overlap=chunk_overlap,
#         length_function=len,
#         separators=["\n\n", "\n", " ", ""]
#     )

#     all_chunks = []

#     for pdf_file in pdf_files:
#         print(f"Processing {pdf_file.name}...")
#         try:
#             loader = PyMuPDFLoader(str(pdf_file))
#             documents = loader.load()

#             # extract structured metadata from filename
#             file_metadata = parse_filename(pdf_file.name)

#             # attach metadata to every page/document
#             for doc in documents:
#                 doc.metadata["source"] = pdf_file.name
#                 doc.metadata["source_type"] = "pdf"
#                 doc.metadata.update(file_metadata)

#             # split into chunks (metadata carries over automatically)
#             chunks = text_splitter.split_documents(documents)
#             all_chunks.extend(chunks)

#         except Exception as e:
#             print(f"Error processing {pdf_file.name}: {e}")

#     print(f"\nProcessed {len(pdf_files)} files into {len(all_chunks)} chunks total")
#     return all_chunks


# chunks = process_all_pdfs(all_pdf_files)

In [5]:
# print(f"Total chunks: {len(chunks)}\n")

# for chunk in chunks[:3]:
#     print("Metadata:", chunk.metadata)
#     print("Content preview:", chunk.page_content[:150])
#     print("-" * 60)

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

semantic_splitter = SemanticChunker(
    embeddings=hf_embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

c:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Dell\AppData\Local\Temp\ipykernel_26628\4137077451.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3426.26it/s]


In [7]:
from langchain_community.document_loaders import PyMuPDFLoader
def process_all_pdfs(pdf_files):
    all_chunks = []

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # extract structured metadata from filename
            file_metadata = parse_filename(pdf_file.name)

            # attach metadata to every page/document
            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["source_type"] = "pdf"
                doc.metadata.update(file_metadata)

               
            chunks = semantic_splitter.split_documents(documents)
            all_chunks.extend(chunks)

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nProcessed {len(pdf_files)} files into {len(all_chunks)} chunks total")
    return all_chunks


chunks = process_all_pdfs(all_pdf_files)
# Semantic Chunking is done okay i make comments of the recursive chunking and if it will not work 
# i will go for the last strategy 

Processing amazon 10-k 2023.pdf...
Processing amazon 10-k 2024.pdf...
Processing amazon 10-q q1 2024.pdf...
Processing amazon 10-q q1 2025.pdf...
Processing amazon 10-q q2 2024.pdf...
Processing amazon 10-q q2 2025.pdf...
Processing amazon 10-q q3 2024.pdf...
Processing apple 10-k 2023.pdf...
Processing apple 10-k 2024.pdf...
Processing apple 10-q q1 2024.pdf...
Processing apple 10-q q2 2024.pdf...
Processing apple 10-q q4 2023.pdf...
Processing apple 8-k q4 2023.pdf...
Processing google 10-k 2023.pdf...
Processing google 10-k 2024 (1).pdf...
Processing google 10-q q1 2025.pdf...
Processing google 10-q q2 2024.pdf...
Processing google 10-q q2 2025.pdf...
Processing google 10-q q3 2024.pdf...
Processing meta 10-q q1 2024.pdf...
Processing meta 10-q q1 2025.pdf...
Processing meta 10-q q2 2024.pdf...
Processing meta 10-q q2 2025.pdf...
Processing meta 10-q q3 2024.pdf...
Processing meta 10-q q3 2025.pdf...
Processing meta 10-q q4 2024.pdf...

Processed 26 files into 3016 chunks total


In [8]:


from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List


# ... rest of your EmbeddingsManager code ...

class EmbeddingsManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name} {e}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_embedding_dimension()


# Initialize it
embeddings_manager = EmbeddingsManager()


Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5009.26it/s]


Model loaded. Embedding dimension: 384


In [9]:
texts = [chunk.page_content for chunk in chunks]
embeddings = embeddings_manager.generate_embeddings(texts)
print(embeddings.shape)  # should be (num_chunks, 384)

Generating embeddings for 3016 texts...


Batches: 100%|██████████| 95/95 [01:21<00:00,  1.17it/s]

Generated embeddings with shape: (3016, 384)
(3016, 384)


In [10]:
import psycopg2

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "rag_chatbot",
    "user": "postgres",
    "password": "taha123" 
}

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
cursor = conn.cursor()
print("Connected to Postgres successfully!")

Connected to Postgres successfully!


In [11]:
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
print("Vector extension ready.")

Vector extension ready.


In [12]:
embedding_dim = embeddings_manager.get_embedding_dimension()  # should print 384
print(f"Embedding dimension: {embedding_dim}")

create_table_query = f"""
CREATE TABLE IF NOT EXISTS document_chunks (
    id SERIAL PRIMARY KEY,
    chunk_id TEXT UNIQUE,
    content TEXT,
    company TEXT,
    filing_type TEXT,
    quarter TEXT,
    year TEXT,
    period TEXT,
    source TEXT,
    embedding vector({embedding_dim})
);
"""
cursor.execute(create_table_query)
print("Table ready.")

Embedding dimension: 384
Table ready.


In [13]:
from psycopg2.extras import execute_values
import uuid

insert_query = """
INSERT INTO document_chunks 
(chunk_id, content, company, filing_type, quarter, year, period, source, embedding)
VALUES %s
ON CONFLICT (chunk_id) DO NOTHING;
"""

rows_to_insert = []
for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    chunk_id = f"chunk_{uuid.uuid4().hex[:8]}_{i}"
    meta = chunk.metadata

    rows_to_insert.append((
        chunk_id,
        chunk.page_content,
        meta.get("company"),
        meta.get("filing_type"),
        meta.get("quarter"),
        meta.get("year"),
        meta.get("period"),
        meta.get("source"),
        embedding.tolist()
    ))

execute_values(cursor, insert_query, rows_to_insert)
print(f"Inserted {len(rows_to_insert)} chunks into Postgres.")

Inserted 3016 chunks into Postgres.


In [14]:
cursor.execute("SELECT COUNT(*) FROM document_chunks;")
print("Total rows in table:", cursor.fetchone()[0])

cursor.execute("SELECT chunk_id, company, filing_type, period, LEFT(content, 100) FROM document_chunks LIMIT 5;")
for row in cursor.fetchall():
    print(row)

Total rows in table: 41008
('chunk_1eb9d950_0', 'amazon', '10-k', '2023', 'Table of Contents\nUNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n\xa0_________')
('chunk_694daf7c_1', 'amazon', '10-k', '2023', '(Address and telephone number, including area code, of registrant’s principal executive offices)\nSec')
('chunk_fbef96db_2', 'amazon', '10-k', '2023', 'Indicate by check mark whether the registrant (1)\xa0has filed all reports required to be filed by Sect')
('chunk_380ff52a_3', 'amazon', '10-k', '2023', 'Indicate by check mark whether the registrant is a large accelerated filer, an accelerated filer, a ')
('chunk_4c55fdb3_4', 'amazon', '10-k', '2023', 'Indicate by check mark whether the registrant has filed a report on and attestation to its managemen')


In [15]:
from typing import List, Dict, Any

class RAGRetriever:
    def __init__(self, conn, embeddings_manager):
        self.conn = conn
        self.embeddings_manager = embeddings_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        query_embedding = self.embeddings_manager.generate_embeddings([query])[0]
        try:
            cur = self.conn.cursor()
            search_query = """
                SELECT chunk_id, content, company, filing_type, quarter, year, period, source,
                       1 - (embedding <=> %s::vector) AS similarity_score
                FROM document_chunks
                ORDER BY embedding <=> %s::vector
                LIMIT %s;
            """
            embedding_list = query_embedding.tolist()
            cur.execute(search_query, (embedding_list, embedding_list, top_k))
            rows = cur.fetchall()

            retrieved_docs = []
            for i, row in enumerate(rows):
                chunk_id, content, company, filing_type, quarter, year, period, source, similarity_score = row
                retrieved_docs.append({
                    'id': chunk_id,
                    'content': content,
                    'metadata': {'company': company, 'filing_type': filing_type, 'quarter': quarter,
                                 'year': year, 'period': period, 'source': source},
                    'similarity_score': similarity_score,
                    'rank': i + 1
                })
            print(f"Retrieved {len(retrieved_docs)} documents")
            return retrieved_docs
        except Exception as e:
            print(f"Error while retrieving documents: {e}")
            return []

rag_retriever = RAGRetriever(conn, embeddings_manager)

In [16]:
results = rag_retriever.retrieve("What was meta's revenue in Q3 2024?", top_k=5)
for r in results:
    print(r['metadata']['company'], r['metadata']['period'], round(r['similarity_score'], 3))
    print(r['content'][:150])
    print("-" * 60)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.70it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
meta Q3 2024 0.59
Meta Earnings Presentation 
Q3 2024
investor.fb.com
------------------------------------------------------------
meta Q3 2024 0.59
Meta Earnings Presentation 
Q3 2024
investor.fb.com
------------------------------------------------------------
meta Q3 2024 0.59
Meta Earnings Presentation 
Q3 2024
investor.fb.com
------------------------------------------------------------
meta Q3 2024 0.59
Meta Earnings Presentation 
Q3 2024
investor.fb.com
------------------------------------------------------------
meta Q3 2024 0.59
Meta Earnings Presentation 
Q3 2024
investor.fb.com
------------------------------------------------------------


Open Ai LLM Connection

In [17]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", openai_api_key is not None)

Key loaded: True


In [18]:
import ollama

response = ollama.chat(
    model="qwen3:4b",
    messages=[{"role": "user", "content": "Say hello in two sentence."}]
)
print(response['message']['content'])

Hello.  
How are you today?


In [19]:
import ollama
from typing import List, Dict, Any

def generate_answer(query: str, retrieved_docs: List[Dict[str, Any]], model: str = "qwen3:4b") -> str:
    if not retrieved_docs:
        return "I couldn't find relevant information to answer that question."

    context_parts = []
    for doc in retrieved_docs:
        meta = doc['metadata']
        source_info = f"[{meta['company'].upper()} {meta['filing_type'].upper()} - {meta['period']}]"
        context_parts.append(f"{source_info}\n{doc['content']}")

    context = "\n\n---\n\n".join(context_parts)

    system_prompt = """You are a financial analyst assistant. Answer the user's question 
using ONLY the provided context from SEC filings. If the context doesn't contain enough 
information to answer, say so clearly. Always cite which company/filing/period your 
answer is based on."""

    user_prompt = f"""Context from SEC filings:

{context}

Question: {query}

Answer based only on the context above:"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        options={"temperature": 0.2}
    )

    return response['message']['content']

In [20]:
def ask_chatbot(query: str, top_k: int = 5) -> str:
    print(f"\nSearching for: {query}\n")
    retrieved_docs = rag_retriever.retrieve(query, top_k=top_k)

    if not retrieved_docs:
        return "No relevant documents found."

    print(f"Using {len(retrieved_docs)} retrieved chunks as context\n")
    answer = generate_answer(query, retrieved_docs)
    return answer

In [21]:
answer = ask_chatbot("What was Apple's revenue in Q1 2024?")
print(answer)


Searching for: What was Apple's revenue in Q1 2024?

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.77it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

Based solely on the provided SEC filings context:  

Apple reported **$119.6 billion** in quarterly revenue for its fiscal 2024 first quarter ended December 30, 2023.  

**Citation**:  
- Apple 8-K filing (Q4 2023) - Exhibit 99.1, as described in all provided excerpts. The context explicitly states: *"The Company posted quarterly revenue of $119.6 billion, up 2 percent year over year"* for the fiscal 2024 first quarter ended December 30, 2023.  

*Note: Apple uses a fiscal year ending in September, so "fiscal 2024 first quarter" corresponds to the calendar period ending December 30, 2023 (as confirmed in the filing).*


In [22]:
answer = ask_chatbot("Amazon FORM 10-K item 1")
print(answer)


Searching for: Amazon FORM 10-K item 1

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.29it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

The provided context contains **no information about Amazon** or any Amazon SEC filings. All context entries are exclusively for **Apple Inc.'s 2024 Form 10-K summaries** (each labeled as "None. Apple Inc. | 2024 Form 10-K | 56"). There are zero Amazon-related filings or references in the provided context.

**Answer**: The context does not contain any Amazon FORM 10-K item 1 information. All provided filings are for Apple Inc. (2024 Form 10-K), and no Amazon filings are included in the context. 

**Citation**: Apple Inc. 2024 Form 10-K (all 5 entries in the context).


In [23]:
answer = ask_chatbot("What was Amazon's total net sales in 2023?")
print(answer)


Searching for: What was Amazon's total net sales in 2023?

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.11it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

Based solely on the provided SEC filings context, Amazon's total net sales for 2023 were **$574,785 million** (as reported in the 10-K filings for the year ended December 31, 2023).  

**Citation**:  
- Amazon 10-K (2024) filing (all four identical excerpts) shows:  
  *"Year Ended December 31, 2023"* → **Consolidated Net Sales: $574,785** (in millions).  

This figure represents the consolidated net sales for the full fiscal year 2023, as defined in the 10-K filing (which reports year-end results). The 10-Q filing (Q1 2024) does not contain 2023 annual data and is irrelevant to this question.


In [24]:
answer = ask_chatbot("What are Amazon's three business segments")
print(answer)


Searching for: What are Amazon's three business segments

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.61it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

Based on the provided SEC filings, Amazon's three business segments are:

North America, International, and Amazon Web Services ("AWS")

This information is consistently stated in multiple Amazon 10-K filings, including both the 2023 and 2024 annual reports. Specifically, the context shows that "We have organized our operations into three segments: North America, International, and Amazon Web Services ("AWS"). These segments reflect the way the Company evaluates its business performance and manages its operations."

The answer is based on Amazon's 10-K filings for 2023 and 2024 (multiple filings are provided in the context).


In [25]:
answer = ask_chatbot("What was Amazon's operating income in 2023?")
print(answer)


Searching for: What was Amazon's operating income in 2023?

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 94.79it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

Based on the provided SEC filings, Amazon's operating income in 2023 was $36.9 billion.

This information is explicitly stated in the Amazon 10-K filing for 2023 (annual report), where it says: "Operating income was $12.2 billion and $36.9 billion for 2022 and 2023."

The filing also shows the consolidated operating income for 2023 as $36,852 million (which rounds to $36.9 billion), consistent with the stated figure.

Answer is based on: Amazon 10-K filing for the year ended December 31, 2023.


In [26]:
answer = ask_chatbot("What was AWS segment's net sales in 2023?")
print(answer)


Searching for: What was AWS segment's net sales in 2023?

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.65it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

Based solely on the provided SEC filings context, the AWS segment's net sales for 2023 were **$90,757 million**.  

This information is explicitly stated in the Amazon 10-K filing for 2023 (repeated across all provided sections), under the "Net Sales" table for the year ended December 31, 2023. The table shows:  
- **AWS**: $90,757 million (2023)  
- **AWS**: $80,096 million (2022)  

The context also confirms this figure aligns with the 13% year-over-year growth rate for AWS (as $80,096 million × 1.13 ≈ $90,757 million).  

**Source**: Amazon 10-K filing for the year ended December 31, 2023 (all repeated sections consistently report this data).


In [27]:
answer = ask_chatbot("Effective tax rate Q1 2024")
print(answer)



Searching for: Effective tax rate Q1 2024

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.31it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

Based solely on the provided SEC filings context:  

**Effective tax rate for Q1 2024 is 13%.**  

This is explicitly stated in the META 10-Q filing for Q3 2025 (all repeated filings show identical data), where the "Q1'24" row under "Effective Tax Rate" lists **13%**.  

*Source: META 10-Q (Q3 2025) filing, "Effective Tax Rate" table, Q1'24 row.*


In [28]:
answer = ask_chatbot("Effective tax rate Q1 2024")
print(answer)


Searching for: Effective tax rate Q1 2024

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.84it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents
Using 5 retrieved chunks as context

Based solely on the provided SEC filings context:  

**Effective tax rate for Q1 2024 is 13%.**  

This is directly stated in the *META 10-Q - Q3 2025* filing (all repeated sections show identical data), where the "Q1'24" column lists an **Effective Tax Rate of 13%**.  

**Citation**: META 10-Q (Q3 2025) filing, "Effective Tax Rate" table, Q1'24 period.
